In [ ]:
import superstats as sup
import pandas as pd

## Constants

In [ ]:
NUM_STEPS = 800

## Prior

In [ ]:
joint_prior = sup.JointPrior(
    v = sup.transition.Mixture(
        transitions=[
            sup.transition.RandomWalk(
                sigma=sup.Prior(dist="halfnormal", scale=0.1),
                delta=0
            ),
            sup.transition.Jump(
                proposal_prior=sup.Prior(dist="logistic", loc=0, scale=1)
            )
        ],
        mixture_weights=sup.Prior(dist="dirichlet", alpha=[20.0, 1.5]),
        bounds=(0.0, 8.0),
        initial_prior=sup.Prior(dist="normal", loc=-1.0, scale=1.0),
    ),
    a = sup.transition.Mixture(
        transitions=[
            sup.transition.RandomWalk(
                sigma=sup.Prior(dist="halfnormal", scale=0.1),
                delta=0
            ),
            sup.transition.Jump(
                proposal_prior=sup.Prior(dist="logistic", loc=0, scale=1)
            )
        ],
        mixture_weights=sup.Prior(dist="dirichlet", alpha=[40.0, 1.5]),
        bounds=(0.0, 6.0),
        initial_prior=sup.Prior(dist="normal", loc=-0.5, scale=1.0),
    ),
    tau = sup.transition.RandomWalk(
        sigma=sup.Prior(dist="halfnormal", scale=0.01),
        delta=0,
        bounds=(0.0, 4.0),
        initial_prior=sup.Prior(dist="normal", loc=-2, scale=1.0)
    ),
    bias = 0.5
)

In [ ]:
fig = joint_prior.plot_joint_prior(num_trajectories=10)

## Model

In [ ]:
ddm = sup.simulation.sample_ddm

model = sup.Model(
    prior=joint_prior,
    simulator=ddm,
)

In [ ]:
fig = model.plot_push_forward(
    num_sim=12,
    num_steps=NUM_STEPS,
    data_dim=0,
    kind="trajectory",
    num_cols=4,
)

## Workflow

In [ ]:
workflow = sup.Workflow(
    model=model,
    checkpoint_filepath="checkpoints/data_application_01"
)

## Training

In [ ]:
train_data = model.sample(
    batch_size=50,
    num_steps=NUM_STEPS,
    tile_to_steps=True
)
test_data = model.sample(
    batch_size=10,
    num_steps=NUM_STEPS,
    tile_to_steps=True
)

In [ ]:
history = workflow.fit_offline(
    data=train_data,
    validation_data=test_data,
    epochs=2,
    batch_size=2
)

In [ ]:
# history = workflow.fit_online(
#     num_steps=NUM_STEPS,
#     epochs=NUM_EPOCHS,
#     num_batches_per_epoch=NUM_ITER_PER_EPOCH,
#     batch_size=BATCH_SIZE
# )

In [ ]:
fig = workflow.plot_history(workflow.history)

## Verification

In [ ]:
val_data = model.sample(
    batch_size=400,
    num_steps=NUM_STEPS
)

In [ ]:
samples = workflow.sample(
    data=val_data,
    num_samples=NUM_SAMPLES,
    inference_batch_size=1
)

### Time-varying Parameters

In [ ]:
fig = workflow.verify_time_varying(val_data, samples)

### Time-invariant Parameters

In [ ]:
fig_recovery, fig_calibration = workflow.verify_time_invariant(
    val_data,
    samples
)

## Data Fitting

In [ ]:
data = pd.read_csv("data/data_color_discrimination.csv")